## Data Preprocessing

The purpose of this notebook is to ingest all flight delay data and create the following training, validation, and test datasets:
    
1. training_data
2. validation_data
3. test_data

These datasets will be used later in the creation of the machine learning models. After running this notebook it will create the 3 files in the parent folder of this notebook.

In [1]:
# Initialization
import pandas as pd
import os
import warnings

warnings.filterwarnings("ignore")
data_dir = r"C:\Users\malop\Documents\CS6220_project\data"

In [2]:
# Load and combine all data
files = [
    file for file in os.listdir(data_dir)
    if file.endswith(".csv")
]

data = pd.concat(
    [pd.read_csv(os.path.join(data_dir, file)) for file in files],
    ignore_index=True
)

In [3]:
# Limit data to major Northeast airport departures (BOS, EWR, JFK, PHL)
airports = ["BOS", "EWR", "JFK", "PHL"]

data = data[data["ORIGIN"].isin(airports)]

In [4]:
# Convert flight date into datetime
data["FL_DATE"] = pd.to_datetime(data["FL_DATE"])

# Initialize holiday feature
data["HOLIDAY_IMPACTED"] = 0

# Define holiday travel windows
holiday_windows = [
    # Thanksgiving: Wednesday before through Sunday after
    ("Thanksgiving", "2024-11-27", "2024-12-01"),
    ("Thanksgiving", "2025-11-26", "2025-11-30"),
    ("Thanksgiving", "2026-11-25", "2026-11-29"),

    # Christmas
    ("Christmas", "2024-12-22", "2024-12-26"),
    ("Christmas", "2025-12-22", "2025-12-26"),
    ("Christmas", "2026-12-22", "2026-12-26"),

    # New Year's
    ("New Year", "2024-12-29", "2025-01-02"),
    ("New Year", "2025-12-29", "2026-01-02"),
    ("New Year", "2026-12-29", "2027-01-02"),

    # Fourth of July
    ("July 4th", "2024-07-02", "2024-07-05"),
    ("July 4th", "2025-07-02", "2025-07-05"),
    ("July 4th", "2026-07-02", "2026-07-05"),

    # Memorial Day
    ("Memorial Day", "2024-05-24", "2024-05-27"),
    ("Memorial Day", "2025-05-23", "2025-05-26"),
    ("Memorial Day", "2026-05-22", "2026-05-25"),

    # Labor Day
    ("Labor Day", "2024-08-30", "2024-09-02"),
    ("Labor Day", "2025-08-29", "2025-09-01"),
    ("Labor Day", "2026-09-04", "2026-09-07")
]

# Flag flights within holiday windows
for holiday, start, end in holiday_windows:
    data.loc[
        data["FL_DATE"].between(start, end),
        "HOLIDAY_IMPACTED"] = 1

# Remove unneeded columns
data = data.drop(columns=["FL_DATE", "TAIL_NUM"])

# Sort data chronologically
data = data.sort_values(
    ["YEAR", "MONTH", "DAY_OF_MONTH"]
).reset_index(drop=True)

# Create historical delay count by airline, flight number, and route
# Looks only at prior flights that occurred prior to flight date, to avoid leakage
data["CARRIER_FL_NUM_DELAYS"] = (
    data.groupby([
        "OP_UNIQUE_CARRIER",
        "OP_CARRIER_FL_NUM",
        "ORIGIN",
        "DEST"
    ])["ARR_DEL15"]
    .transform(lambda x: x.shift().cumsum())
)

# Replace missing historical counts with 0
data["CARRIER_FL_NUM_DELAYS"] = data["CARRIER_FL_NUM_DELAYS"].fillna(0)

# One-hot encode origin and destination airports
data = pd.get_dummies(
    data,
    columns=["ORIGIN", "DEST"],
    dtype=int
)

# One-hot encode airline carriers
data = pd.get_dummies(data, columns=["OP_UNIQUE_CARRIER"], dtype=int)

# Drop flight number
data = data.drop(columns=["OP_CARRIER_FL_NUM"])

# Drop redundant city and state columns
data = data.drop(columns=[
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_ABR",
    "DEST_CITY_NAME",
    "DEST_STATE_ABR"
])

# Drop due to leakaage
data = data.drop(columns=["DEP_TIME", "DEP_DELAY"])

# Extract SCHEDULED departure hour and minute (no leakage)
data["CRS_DEP_HOUR"] = data["CRS_DEP_TIME"] // 100
data["CRS_DEP_MINUTE"] = data["CRS_DEP_TIME"] % 100

# Drop original column
data = data.drop(columns=["CRS_DEP_TIME"])

# Drop redundant column
data = data.drop(columns=["DEP_TIME_BLK"])

# Extract SCHEDULED arrival time into hour and minute (no leakage)
data["CRS_ARR_HOUR"] = data["CRS_ARR_TIME"] // 100
data["CRS_ARR_MINUTE"] = data["CRS_ARR_TIME"] % 100

# Drop unnecessary and leakage columns
data = data.drop(columns=[
    "CRS_ARR_TIME",
    "ARR_TIME",
    "ARR_DELAY"
])

# Drop redundant column
data = data.drop(columns=["ARR_TIME_BLK"])

# Drop cancellation, diversion, and post-flight delay information (leakage)
data = data.drop(columns=[
    "CANCELLED",
    "CANCELLATION_CODE",
    "DIVERTED",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
])

# Drop rows with missing target values
data = data.dropna(subset=["ARR_DEL15"]).reset_index(drop=True)

# Add departure time period
data["CRS_DEP_PERIOD"] = pd.cut(
    data["CRS_DEP_HOUR"],
    bins=[-1, 4, 10, 16, 20, 23],
    labels=["Night", "Morning", "Afternoon", "Evening", "Night"],
    ordered=False
)

# One-hot encode departure time period
data = pd.get_dummies(
    data,
    columns=["CRS_DEP_PERIOD"],
    dtype=int
)

In [5]:
# Create training data (January 2024 to June 2025)
train_data = data[
    ((data["YEAR"] == 2024) & (data["MONTH"] >= 1)) |
    ((data["YEAR"] == 2025) & (data["MONTH"] <= 6))
]

# Create validation data (July 2025 to December 2025)
val_data = data[
    (data["YEAR"] == 2025) &
    (data["MONTH"] >= 7) &
    (data["MONTH"] <= 12)
]

# Create test data (January 2026 to August 2026)
test_data = data[
    (data["YEAR"] == 2026) &
    (data["MONTH"] >= 1) &
    (data["MONTH"] <= 8)
]

In [6]:
# Display all columns
pd.set_option("display.max_columns", None)

# View first 5 rows of each dataset
print("Training Data")
display(train_data.head())
display(train_data.sample(5))

print("Validation Data")
display(val_data.head())
display(val_data.sample(5))

print("Test Data")
display(test_data.head())
display(test_data.sample(5))

Training Data


,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,HOLIDAY_IMPACTED,CARRIER_FL_NUM_DELAYS,ORIGIN_BOS,ORIGIN_EWR,ORIGIN_JFK,ORIGIN_PHL,DEST_ABQ,DEST_ACK,DEST_ACY,DEST_AGS,DEST_ALB,DEST_ANC,DEST_ATL,DEST_ATW,DEST_AUS,DEST_AVL,DEST_BDL,DEST_BGR,DEST_BHM,DEST_BNA,DEST_BOS,DEST_BQN,DEST_BTV,DEST_BUF,DEST_BUR,DEST_BWI,DEST_BZN,DEST_CAE,DEST_CHA,DEST_CHS,DEST_CID,DEST_CLE,DEST_CLT,DEST_CMH,DEST_CVG,DEST_DAB,DEST_DAL,DEST_DAY,DEST_DCA,DEST_DEN,DEST_DFW,DEST_DJT,DEST_DSM,DEST_DTW,DEST_ECP,DEST_EGE,DEST_EWR,DEST_EYW,DEST_FCA,DEST_FLL,DEST_FNT,DEST_GRB,DEST_GRR,DEST_GSO,DEST_GSP,DEST_HDN,DEST_HHH,DEST_HNL,DEST_HOU,DEST_HYA,DEST_IAD,DEST_IAH,DEST_ILM,DEST_IND,DEST_ISP,DEST_ITH,DEST_JAC,DEST_JAX,DEST_JFK,DEST_LAS,DEST_LAX,DEST_LEX,DEST_LGA,DEST_MCI,DEST_MCO,DEST_MDT,DEST_MDW,DEST_MEM,DEST_MHT,DEST_MIA,DEST_MKE,DEST_MSN,DEST_MSP,DEST_MSY,DEST_MTJ,DEST_MVY,DEST_MYR,DEST_OAK,DEST_OGG,DEST_OMA,DEST_ONT,DEST_ORD,DEST_ORF,DEST_ORH,DEST_PBI,DEST_PDX,DEST_PHL,DEST_PHX,DEST_PIT,DEST_PNS,DEST_PQI,DEST_PSE,DEST_PSP,DEST_PVD,DEST_PWM,DEST_RDU,DEST_RIC,DEST_RNO,DEST_ROA,DEST_ROC,DEST_RSW,DEST_SAN,DEST_SAT,DEST_SAV,DEST_SBN,DEST_SCE,DEST_SDF,DEST_SEA,DEST_SFO,DEST_SJC,DEST_SJU,DEST_SLC,DEST_SMF,DEST_SNA,DEST_SRQ,DEST_STL,DEST_STT,DEST_SYR,DEST_TPA,DEST_TTN,DEST_TVC,DEST_TYS,DEST_VPS,DEST_VRB,DEST_XNA,OP_UNIQUE_CARRIER_9E,OP_UNIQUE_CARRIER_AA,OP_UNIQUE_CARRIER_AS,OP_UNIQUE_CARRIER_B6,OP_UNIQUE_CARRIER_DL,OP_UNIQUE_CARRIER_F9,OP_UNIQUE_CARRIER_G4,OP_UNIQUE_CARRIER_HA,OP_UNIQUE_CARRIER_MQ,OP_UNIQUE_CARRIER_NK,OP_UNIQUE_CARRIER_OH,OP_UNIQUE_CARRIER_OO,OP_UNIQUE_CARRIER_UA,OP_UNIQUE_CARRIER_WN,OP_UNIQUE_CARRIER_YX,CRS_DEP_HOUR,CRS_DEP_MINUTE,CRS_ARR_HOUR,CRS_ARR_MINUTE,CRS_DEP_PERIOD_Afternoon,CRS_DEP_PERIOD_Evening,CRS_DEP_PERIOD_Morning,CRS_DEP_PERIOD_Night
0,2024,1,1,1,0.0,162.0,752.0,0,0.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,5,16,47,1,0,0,0
1,2024,1,1,1,0.0,71.0,189.0,0,0.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,4,12,15,1,0,0,0
2,2024,1,1,1,0.0,100.0,264.0,0,0.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16,20,18,0,1,0,0,0
3,2024,1,1,1,0.0,108.0,301.0,0,0.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,19,50,21,38,0,1,0,0
4,2024,1,1,1,0.0,166.0,765.0,0,0.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,19,46,0,1,0,0


,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,HOLIDAY_IMPACTED,CARRIER_FL_NUM_DELAYS,ORIGIN_BOS,ORIGIN_EWR,ORIGIN_JFK,ORIGIN_PHL,DEST_ABQ,DEST_ACK,DEST_ACY,DEST_AGS,DEST_ALB,DEST_ANC,DEST_ATL,DEST_ATW,DEST_AUS,DEST_AVL,DEST_BDL,DEST_BGR,DEST_BHM,DEST_BNA,DEST_BOS,DEST_BQN,DEST_BTV,DEST_BUF,DEST_BUR,DEST_BWI,DEST_BZN,DEST_CAE,DEST_CHA,DEST_CHS,DEST_CID,DEST_CLE,DEST_CLT,DEST_CMH,DEST_CVG,DEST_DAB,DEST_DAL,DEST_DAY,DEST_DCA,DEST_DEN,DEST_DFW,DEST_DJT,DEST_DSM,DEST_DTW,DEST_ECP,DEST_EGE,DEST_EWR,DEST_EYW,DEST_FCA,DEST_FLL,DEST_FNT,DEST_GRB,DEST_GRR,DEST_GSO,DEST_GSP,DEST_HDN,DEST_HHH,DEST_HNL,DEST_HOU,DEST_HYA,DEST_IAD,DEST_IAH,DEST_ILM,DEST_IND,DEST_ISP,DEST_ITH,DEST_JAC,DEST_JAX,DEST_JFK,DEST_LAS,DEST_LAX,DEST_LEX,DEST_LGA,DEST_MCI,DEST_MCO,DEST_MDT,DEST_MDW,DEST_MEM,DEST_MHT,DEST_MIA,DEST_MKE,DEST_MSN,DEST_MSP,DEST_MSY,DEST_MTJ,DEST_MVY,DEST_MYR,DEST_OAK,DEST_OGG,DEST_OMA,DEST_ONT,DEST_ORD,DEST_ORF,DEST_ORH,DEST_PBI,DEST_PDX,DEST_PHL,DEST_PHX,DEST_PIT,DEST_PNS,DEST_PQI,DEST_PSE,DEST_PSP,DEST_PVD,DEST_PWM,DEST_RDU,DEST_RIC,DEST_RNO,DEST_ROA,DEST_ROC,DEST_RSW,DEST_SAN,DEST_SAT,DEST_SAV,DEST_SBN,DEST_SCE,DEST_SDF,DEST_SEA,DEST_SFO,DEST_SJC,DEST_SJU,DEST_SLC,DEST_SMF,DEST_SNA,DEST_SRQ,DEST_STL,DEST_STT,DEST_SYR,DEST_TPA,DEST_TTN,DEST_TVC,DEST_TYS,DEST_VPS,DEST_VRB,DEST_XNA,OP_UNIQUE_CARRIER_9E,OP_UNIQUE_CARRIER_AA,OP_UNIQUE_CARRIER_AS,OP_UNIQUE_CARRIER_B6,OP_UNIQUE_CARRIER_DL,OP_UNIQUE_CARRIER_F9,OP_UNIQUE_CARRIER_G4,OP_UNIQUE_CARRIER_HA,OP_UNIQUE_CARRIER_MQ,OP_UNIQUE_CARRIER_NK,OP_UNIQUE_CARRIER_OH,OP_UNIQUE_CARRIER_OO,OP_UNIQUE_CARRIER_UA,OP_UNIQUE_CARRIER_WN,OP_UNIQUE_CARRIER_YX,CRS_DEP_HOUR,CRS_DEP_MINUTE,CRS_ARR_HOUR,CRS_ARR_MINUTE,CRS_DEP_PERIOD_Afternoon,CRS_DEP_PERIOD_Evening,CRS_DEP_PERIOD_Morning,CRS_DEP_PERIOD_Night
646197,2025,5,12,1,0.0,119.0,550.0,0,38.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,7,10,9,9,0,0,1,0
708244,2025,6,30,1,0.0,95.0,213.0,0,89.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,9,8,10,43,0,0,1,0
565590,2025,3,10,1,1.0,384.0,2475.0,0,49.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,15,50,19,14,1,0,0,0
77659,2024,3,6,3,0.0,165.0,719.0,0,6.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,21,0,22,45,0,0,0,1
262552,2024,7,21,7,0.0,167.0,920.0,0,34.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,18,6,20,53,0,1,0,0


Validation Data


,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,HOLIDAY_IMPACTED,CARRIER_FL_NUM_DELAYS,ORIGIN_BOS,ORIGIN_EWR,ORIGIN_JFK,ORIGIN_PHL,DEST_ABQ,DEST_ACK,DEST_ACY,DEST_AGS,DEST_ALB,DEST_ANC,DEST_ATL,DEST_ATW,DEST_AUS,DEST_AVL,DEST_BDL,DEST_BGR,DEST_BHM,DEST_BNA,DEST_BOS,DEST_BQN,DEST_BTV,DEST_BUF,DEST_BUR,DEST_BWI,DEST_BZN,DEST_CAE,DEST_CHA,DEST_CHS,DEST_CID,DEST_CLE,DEST_CLT,DEST_CMH,DEST_CVG,DEST_DAB,DEST_DAL,DEST_DAY,DEST_DCA,DEST_DEN,DEST_DFW,DEST_DJT,DEST_DSM,DEST_DTW,DEST_ECP,DEST_EGE,DEST_EWR,DEST_EYW,DEST_FCA,DEST_FLL,DEST_FNT,DEST_GRB,DEST_GRR,DEST_GSO,DEST_GSP,DEST_HDN,DEST_HHH,DEST_HNL,DEST_HOU,DEST_HYA,DEST_IAD,DEST_IAH,DEST_ILM,DEST_IND,DEST_ISP,DEST_ITH,DEST_JAC,DEST_JAX,DEST_JFK,DEST_LAS,DEST_LAX,DEST_LEX,DEST_LGA,DEST_MCI,DEST_MCO,DEST_MDT,DEST_MDW,DEST_MEM,DEST_MHT,DEST_MIA,DEST_MKE,DEST_MSN,DEST_MSP,DEST_MSY,DEST_MTJ,DEST_MVY,DEST_MYR,DEST_OAK,DEST_OGG,DEST_OMA,DEST_ONT,DEST_ORD,DEST_ORF,DEST_ORH,DEST_PBI,DEST_PDX,DEST_PHL,DEST_PHX,DEST_PIT,DEST_PNS,DEST_PQI,DEST_PSE,DEST_PSP,DEST_PVD,DEST_PWM,DEST_RDU,DEST_RIC,DEST_RNO,DEST_ROA,DEST_ROC,DEST_RSW,DEST_SAN,DEST_SAT,DEST_SAV,DEST_SBN,DEST_SCE,DEST_SDF,DEST_SEA,DEST_SFO,DEST_SJC,DEST_SJU,DEST_SLC,DEST_SMF,DEST_SNA,DEST_SRQ,DEST_STL,DEST_STT,DEST_SYR,DEST_TPA,DEST_TTN,DEST_TVC,DEST_TYS,DEST_VPS,DEST_VRB,DEST_XNA,OP_UNIQUE_CARRIER_9E,OP_UNIQUE_CARRIER_AA,OP_UNIQUE_CARRIER_AS,OP_UNIQUE_CARRIER_B6,OP_UNIQUE_CARRIER_DL,OP_UNIQUE_CARRIER_F9,OP_UNIQUE_CARRIER_G4,OP_UNIQUE_CARRIER_HA,OP_UNIQUE_CARRIER_MQ,OP_UNIQUE_CARRIER_NK,OP_UNIQUE_CARRIER_OH,OP_UNIQUE_CARRIER_OO,OP_UNIQUE_CARRIER_UA,OP_UNIQUE_CARRIER_WN,OP_UNIQUE_CARRIER_YX,CRS_DEP_HOUR,CRS_DEP_MINUTE,CRS_ARR_HOUR,CRS_ARR_MINUTE,CRS_DEP_PERIOD_Afternoon,CRS_DEP_PERIOD_Evening,CRS_DEP_PERIOD_Morning,CRS_DEP_PERIOD_Night
708291,2025,7,1,2,0.0,379.0,2586.0,0,34.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,7,55,11,14,0,0,1,0
708292,2025,7,1,2,0.0,93.0,280.0,0,18.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,5,49,7,22,0,0,1,0
708293,2025,7,1,2,0.0,109.0,449.0,0,67.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,8,20,10,9,0,0,1,0
708294,2025,7,1,2,1.0,370.0,2475.0,0,0.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,17,49,20,59,0,1,0,0
708295,2025,7,1,2,1.0,367.0,2475.0,0,77.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,16,29,19,36,1,0,0,0


,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,HOLIDAY_IMPACTED,CARRIER_FL_NUM_DELAYS,ORIGIN_BOS,ORIGIN_EWR,ORIGIN_JFK,ORIGIN_PHL,DEST_ABQ,DEST_ACK,DEST_ACY,DEST_AGS,DEST_ALB,DEST_ANC,DEST_ATL,DEST_ATW,DEST_AUS,DEST_AVL,DEST_BDL,DEST_BGR,DEST_BHM,DEST_BNA,DEST_BOS,DEST_BQN,DEST_BTV,DEST_BUF,DEST_BUR,DEST_BWI,DEST_BZN,DEST_CAE,DEST_CHA,DEST_CHS,DEST_CID,DEST_CLE,DEST_CLT,DEST_CMH,DEST_CVG,DEST_DAB,DEST_DAL,DEST_DAY,DEST_DCA,DEST_DEN,DEST_DFW,DEST_DJT,DEST_DSM,DEST_DTW,DEST_ECP,DEST_EGE,DEST_EWR,DEST_EYW,DEST_FCA,DEST_FLL,DEST_FNT,DEST_GRB,DEST_GRR,DEST_GSO,DEST_GSP,DEST_HDN,DEST_HHH,DEST_HNL,DEST_HOU,DEST_HYA,DEST_IAD,DEST_IAH,DEST_ILM,DEST_IND,DEST_ISP,DEST_ITH,DEST_JAC,DEST_JAX,DEST_JFK,DEST_LAS,DEST_LAX,DEST_LEX,DEST_LGA,DEST_MCI,DEST_MCO,DEST_MDT,DEST_MDW,DEST_MEM,DEST_MHT,DEST_MIA,DEST_MKE,DEST_MSN,DEST_MSP,DEST_MSY,DEST_MTJ,DEST_MVY,DEST_MYR,DEST_OAK,DEST_OGG,DEST_OMA,DEST_ONT,DEST_ORD,DEST_ORF,DEST_ORH,DEST_PBI,DEST_PDX,DEST_PHL,DEST_PHX,DEST_PIT,DEST_PNS,DEST_PQI,DEST_PSE,DEST_PSP,DEST_PVD,DEST_PWM,DEST_RDU,DEST_RIC,DEST_RNO,DEST_ROA,DEST_ROC,DEST_RSW,DEST_SAN,DEST_SAT,DEST_SAV,DEST_SBN,DEST_SCE,DEST_SDF,DEST_SEA,DEST_SFO,DEST_SJC,DEST_SJU,DEST_SLC,DEST_SMF,DEST_SNA,DEST_SRQ,DEST_STL,DEST_STT,DEST_SYR,DEST_TPA,DEST_TTN,DEST_TVC,DEST_TYS,DEST_VPS,DEST_VRB,DEST_XNA,OP_UNIQUE_CARRIER_9E,OP_UNIQUE_CARRIER_AA,OP_UNIQUE_CARRIER_AS,OP_UNIQUE_CARRIER_B6,OP_UNIQUE_CARRIER_DL,OP_UNIQUE_CARRIER_F9,OP_UNIQUE_CARRIER_G4,OP_UNIQUE_CARRIER_HA,OP_UNIQUE_CARRIER_MQ,OP_UNIQUE_CARRIER_NK,OP_UNIQUE_CARRIER_OH,OP_UNIQUE_CARRIER_OO,OP_UNIQUE_CARRIER_UA,OP_UNIQUE_CARRIER_WN,OP_UNIQUE_CARRIER_YX,CRS_DEP_HOUR,CRS_DEP_MINUTE,CRS_ARR_HOUR,CRS_ARR_MINUTE,CRS_DEP_PERIOD_Afternoon,CRS_DEP_PERIOD_Evening,CRS_DEP_PERIOD_Morning,CRS_DEP_PERIOD_Night
772403,2025,8,20,3,1.0,245.0,1576.0,0,7.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,18,44,22,49,0,1,0,0
813997,2025,9,21,7,0.0,93.0,404.0,0,6.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,14,29,16,2,1,0,0,0
802028,2025,9,12,5,1.0,112.0,395.0,0,138.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,20,46,22,38,0,1,0,0
900038,2025,11,28,5,0.0,178.0,944.0,1,30.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,13,0,15,58,1,0,0,0
840719,2025,10,12,7,1.0,159.0,783.0,0,26.0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,8,10,10,49,0,0,1,0


Test Data


,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,HOLIDAY_IMPACTED,CARRIER_FL_NUM_DELAYS,ORIGIN_BOS,ORIGIN_EWR,ORIGIN_JFK,ORIGIN_PHL,DEST_ABQ,DEST_ACK,DEST_ACY,DEST_AGS,DEST_ALB,DEST_ANC,DEST_ATL,DEST_ATW,DEST_AUS,DEST_AVL,DEST_BDL,DEST_BGR,DEST_BHM,DEST_BNA,DEST_BOS,DEST_BQN,DEST_BTV,DEST_BUF,DEST_BUR,DEST_BWI,DEST_BZN,DEST_CAE,DEST_CHA,DEST_CHS,DEST_CID,DEST_CLE,DEST_CLT,DEST_CMH,DEST_CVG,DEST_DAB,DEST_DAL,DEST_DAY,DEST_DCA,DEST_DEN,DEST_DFW,DEST_DJT,DEST_DSM,DEST_DTW,DEST_ECP,DEST_EGE,DEST_EWR,DEST_EYW,DEST_FCA,DEST_FLL,DEST_FNT,DEST_GRB,DEST_GRR,DEST_GSO,DEST_GSP,DEST_HDN,DEST_HHH,DEST_HNL,DEST_HOU,DEST_HYA,DEST_IAD,DEST_IAH,DEST_ILM,DEST_IND,DEST_ISP,DEST_ITH,DEST_JAC,DEST_JAX,DEST_JFK,DEST_LAS,DEST_LAX,DEST_LEX,DEST_LGA,DEST_MCI,DEST_MCO,DEST_MDT,DEST_MDW,DEST_MEM,DEST_MHT,DEST_MIA,DEST_MKE,DEST_MSN,DEST_MSP,DEST_MSY,DEST_MTJ,DEST_MVY,DEST_MYR,DEST_OAK,DEST_OGG,DEST_OMA,DEST_ONT,DEST_ORD,DEST_ORF,DEST_ORH,DEST_PBI,DEST_PDX,DEST_PHL,DEST_PHX,DEST_PIT,DEST_PNS,DEST_PQI,DEST_PSE,DEST_PSP,DEST_PVD,DEST_PWM,DEST_RDU,DEST_RIC,DEST_RNO,DEST_ROA,DEST_ROC,DEST_RSW,DEST_SAN,DEST_SAT,DEST_SAV,DEST_SBN,DEST_SCE,DEST_SDF,DEST_SEA,DEST_SFO,DEST_SJC,DEST_SJU,DEST_SLC,DEST_SMF,DEST_SNA,DEST_SRQ,DEST_STL,DEST_STT,DEST_SYR,DEST_TPA,DEST_TTN,DEST_TVC,DEST_TYS,DEST_VPS,DEST_VRB,DEST_XNA,OP_UNIQUE_CARRIER_9E,OP_UNIQUE_CARRIER_AA,OP_UNIQUE_CARRIER_AS,OP_UNIQUE_CARRIER_B6,OP_UNIQUE_CARRIER_DL,OP_UNIQUE_CARRIER_F9,OP_UNIQUE_CARRIER_G4,OP_UNIQUE_CARRIER_HA,OP_UNIQUE_CARRIER_MQ,OP_UNIQUE_CARRIER_NK,OP_UNIQUE_CARRIER_OH,OP_UNIQUE_CARRIER_OO,OP_UNIQUE_CARRIER_UA,OP_UNIQUE_CARRIER_WN,OP_UNIQUE_CARRIER_YX,CRS_DEP_HOUR,CRS_DEP_MINUTE,CRS_ARR_HOUR,CRS_ARR_MINUTE,CRS_DEP_PERIOD_Afternoon,CRS_DEP_PERIOD_Evening,CRS_DEP_PERIOD_Morning,CRS_DEP_PERIOD_Night
940573,2026,1,1,4,1.0,382.0,2475.0,1,52.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,9,41,13,3,0,0,1,0
940574,2026,1,1,4,1.0,375.0,2475.0,1,160.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,17,45,21,0,0,1,0,0
940575,2026,1,1,4,0.0,374.0,2454.0,1,12.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,9,25,12,39,0,0,1,0
940576,2026,1,1,4,1.0,400.0,2586.0,1,48.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,10,30,14,10,0,0,1,0
940577,2026,1,1,4,0.0,138.0,742.0,1,6.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,15,14,17,32,1,0,0,0


,YEAR,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ARR_DEL15,CRS_ELAPSED_TIME,DISTANCE,HOLIDAY_IMPACTED,CARRIER_FL_NUM_DELAYS,ORIGIN_BOS,ORIGIN_EWR,ORIGIN_JFK,ORIGIN_PHL,DEST_ABQ,DEST_ACK,DEST_ACY,DEST_AGS,DEST_ALB,DEST_ANC,DEST_ATL,DEST_ATW,DEST_AUS,DEST_AVL,DEST_BDL,DEST_BGR,DEST_BHM,DEST_BNA,DEST_BOS,DEST_BQN,DEST_BTV,DEST_BUF,DEST_BUR,DEST_BWI,DEST_BZN,DEST_CAE,DEST_CHA,DEST_CHS,DEST_CID,DEST_CLE,DEST_CLT,DEST_CMH,DEST_CVG,DEST_DAB,DEST_DAL,DEST_DAY,DEST_DCA,DEST_DEN,DEST_DFW,DEST_DJT,DEST_DSM,DEST_DTW,DEST_ECP,DEST_EGE,DEST_EWR,DEST_EYW,DEST_FCA,DEST_FLL,DEST_FNT,DEST_GRB,DEST_GRR,DEST_GSO,DEST_GSP,DEST_HDN,DEST_HHH,DEST_HNL,DEST_HOU,DEST_HYA,DEST_IAD,DEST_IAH,DEST_ILM,DEST_IND,DEST_ISP,DEST_ITH,DEST_JAC,DEST_JAX,DEST_JFK,DEST_LAS,DEST_LAX,DEST_LEX,DEST_LGA,DEST_MCI,DEST_MCO,DEST_MDT,DEST_MDW,DEST_MEM,DEST_MHT,DEST_MIA,DEST_MKE,DEST_MSN,DEST_MSP,DEST_MSY,DEST_MTJ,DEST_MVY,DEST_MYR,DEST_OAK,DEST_OGG,DEST_OMA,DEST_ONT,DEST_ORD,DEST_ORF,DEST_ORH,DEST_PBI,DEST_PDX,DEST_PHL,DEST_PHX,DEST_PIT,DEST_PNS,DEST_PQI,DEST_PSE,DEST_PSP,DEST_PVD,DEST_PWM,DEST_RDU,DEST_RIC,DEST_RNO,DEST_ROA,DEST_ROC,DEST_RSW,DEST_SAN,DEST_SAT,DEST_SAV,DEST_SBN,DEST_SCE,DEST_SDF,DEST_SEA,DEST_SFO,DEST_SJC,DEST_SJU,DEST_SLC,DEST_SMF,DEST_SNA,DEST_SRQ,DEST_STL,DEST_STT,DEST_SYR,DEST_TPA,DEST_TTN,DEST_TVC,DEST_TYS,DEST_VPS,DEST_VRB,DEST_XNA,OP_UNIQUE_CARRIER_9E,OP_UNIQUE_CARRIER_AA,OP_UNIQUE_CARRIER_AS,OP_UNIQUE_CARRIER_B6,OP_UNIQUE_CARRIER_DL,OP_UNIQUE_CARRIER_F9,OP_UNIQUE_CARRIER_G4,OP_UNIQUE_CARRIER_HA,OP_UNIQUE_CARRIER_MQ,OP_UNIQUE_CARRIER_NK,OP_UNIQUE_CARRIER_OH,OP_UNIQUE_CARRIER_OO,OP_UNIQUE_CARRIER_UA,OP_UNIQUE_CARRIER_WN,OP_UNIQUE_CARRIER_YX,CRS_DEP_HOUR,CRS_DEP_MINUTE,CRS_ARR_HOUR,CRS_ARR_MINUTE,CRS_DEP_PERIOD_Afternoon,CRS_DEP_PERIOD_Evening,CRS_DEP_PERIOD_Morning,CRS_DEP_PERIOD_Night
1115605,2026,5,27,3,0.0,350.0,2248.0,0,0.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,7,50,10,40,0,0,1,0
960551,2026,1,18,7,1.0,147.0,708.0,0,1.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,9,55,12,22,0,0,1,0
1048799,2026,4,6,1,0.0,255.0,1391.0,0,58.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,8,25,11,40,0,0,1,0
1114262,2026,5,26,2,0.0,170.0,951.0,0,68.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,20,50,23,40,0,1,0,0
1106577,2026,5,20,3,0.0,194.0,1089.0,0,41.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,10,15,13,29,0,0,1,0


In [7]:
# Data set sizes
print("Combined:", data.shape)
print("Training:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Combined: (1241715, 165)
Training: (708291, 165)
Validation: (232282, 165)
Test: (301142, 165)


In [8]:
# Get current working directory
output_dir = os.getcwd()

# Save datasets as CSV
train_data.to_csv(os.path.join(output_dir, "train_data.csv"), index=False)
val_data.to_csv(os.path.join(output_dir, "val_data.csv"), index=False)
test_data.to_csv(os.path.join(output_dir, "test_data.csv"), index=False)

print("Datasets saved to:", output_dir)

Datasets saved to: C:\Users\malop\Documents\CS6220_project
